# Meaning as a position

MichAl Academy, lesson 3.8.

Run each cell with **Shift+Enter**.

An embedding gives every item a position, arranged so that similar items land in
similar places. Then "find me things like this" becomes a distance calculation.

Everything here is built from the corpus in front of us. No pretrained model is
downloaded, which is the point: every number below came from text you could read.

**This notebook fetches the 20 newsgroups dataset the first time it runs**, about
14 MB, because no bundled dataset has real topic labels on real prose.


In [ ]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

np.set_printoptions(precision=4, suppress=True)

CATS = ['comp.graphics', 'rec.sport.hockey', 'sci.med', 'talk.politics.guns',
        'sci.space', 'rec.autos']
data = fetch_20newsgroups(subset='train', categories=CATS,
                          remove=('headers', 'footers', 'quotes'), random_state=0)

keep = [i for i, d in enumerate(data.data) if len(d.split()) >= 20]
docs = [data.data[i] for i in keep]
y = np.array(data.target)[keep]

print(f"documents: {len(docs)} over {len(CATS)} topics")
print(f"median length: {int(np.median([len(d.split()) for d in docs]))} words")


In [ ]:
tfidf = TfidfVectorizer(min_df=5, max_df=0.5, stop_words='english')
Xt = tfidf.fit_transform(docs)
print(f"tf-idf matrix: {Xt.shape}, {100 * Xt.nnz / (Xt.shape[0] * Xt.shape[1]):.4f}% non-zero")


## 1. Do similar documents land together?

The test is simple: for each post, is its nearest neighbour from the same topic?


In [ ]:
def neighbour_accuracy(V, labels, k=1):
    """Cosine: normalise, then the dot product IS the similarity."""
    M = normalize(V)
    S = M @ M.T
    np.fill_diagonal(S, -np.inf)
    idx = np.argpartition(-S, k, axis=1)[:, :k]
    return float((labels[idx] == labels[:, None]).mean())


def euclid_neighbour_accuracy(V, labels, k=1):
    M = np.asarray(V)
    d2 = (M ** 2).sum(1)[:, None] + (M ** 2).sum(1)[None, :] - 2 * (M @ M.T)
    np.fill_diagonal(d2, np.inf)
    idx = np.argpartition(d2, k, axis=1)[:, :k]
    return float((labels[idx] == labels[:, None]).mean())


svd = TruncatedSVD(n_components=100, random_state=0)
E = svd.fit_transform(Xt)

print(f"guessing, with {len(CATS)} topics    : {1 / len(CATS):.4f}")
print(f"nearest neighbour shares topic : {neighbour_accuracy(E, y):.4f}")
print(f"variance kept by 100 dimensions: {svd.explained_variance_ratio_.sum():.4f}")


Nobody supplied the topics. The method arranged posts by which words they use and
the topics fell out of that arrangement.

## 2. Direction is the meaning, length is not

This is the fact that decides whether retrieval works. Compare documents by raw
word counts, where documents differ in length.


In [ ]:
counts = CountVectorizer(min_df=5, max_df=0.5, stop_words='english').fit_transform(docs)
craw = np.asarray(counts.todense()).astype(np.float64)
lengths = craw.sum(axis=1)

print(f"document length: median {np.median(lengths):.0f}, "
      f"10th pct {np.percentile(lengths, 10):.0f}, 90th pct {np.percentile(lengths, 90):.0f}")
print(f"  cosine    : {neighbour_accuracy(craw, y):.4f}")
print(f"  euclidean : {euclid_neighbour_accuracy(craw, y):.4f}")


In [ ]:
# How much of euclidean distance is just a statement about length?
sub = np.random.default_rng(0).choice(len(craw), 400, replace=False)
M = craw[sub]
d2 = (M ** 2).sum(1)[:, None] + (M ** 2).sum(1)[None, :] - 2 * (M @ M.T)
dist = np.sqrt(np.maximum(d2, 0))
ldiff = np.abs(lengths[sub][:, None] - lengths[sub][None, :])
iu = np.triu_indices(len(sub), 1)

print("correlation between euclidean distance and pure length difference: "
      f"{np.corrcoef(dist[iu], ldiff[iu])[0, 1]:.4f}")


Euclidean distance on raw counts is, to a first approximation, measuring which
document is longer. Cosine ignores length and compares direction only.

Now the case that looks like a contradiction.


In [ ]:
raw_tfidf = np.asarray(Xt.todense())
print(f"on tf-idf, cosine    : {neighbour_accuracy(raw_tfidf, y):.4f}")
print(f"on tf-idf, euclidean : {euclid_neighbour_accuracy(raw_tfidf, y):.4f}")
print(f"row norms of tf-idf: min {np.linalg.norm(raw_tfidf, axis=1).min():.4f}, "
      f"max {np.linalg.norm(raw_tfidf, axis=1).max():.4f}")


Identical, because scikit-learn's tf-idf is already scaled to unit length. Once
every vector has the same length, ranking by distance and ranking by angle are
the same ranking.

So the rule is not "cosine is magic". It is **normalise, and then the choice
stops mattering**.

## 3. How many dimensions?


In [ ]:
print(f"{'dimensions':<14}{'neighbour@1':<14}variance kept")
for d in (2, 5, 10, 25, 50, 100, 300):
    s = TruncatedSVD(n_components=d, random_state=0)
    V = s.fit_transform(Xt)
    print(f"{d:<14}{neighbour_accuracy(V, y):<14.4f}{s.explained_variance_ratio_.sum():.4f}")
print(f"{'full tf-idf':<14}{neighbour_accuracy(raw_tfidf, y):<14.4f}1.0000")


Two lessons in one table.

Ten numbers do almost all the work, and going to a hundred is ten times the
storage for slightly worse retrieval.

And **variance kept is the wrong thing to optimise**. The best compressed result
keeps only a few percent of the variance. Choosing dimensions by the usual advice
of retaining 90% would have taken far more of them and done worse, because the
extra directions are real variance describing writing style, length and quoting
habits rather than subject.

## 4. What an embedding actually contains

Word vectors now, from which words share a document.


In [ ]:
cv = CountVectorizer(min_df=20, max_df=0.3, stop_words='english')
C = cv.fit_transform(docs)
vocab = np.array(cv.get_feature_names_out())
print(f"vocabulary: {len(vocab)} words")

CO = (C.T @ C).toarray().astype(np.float64)
np.fill_diagonal(CO, 0)
W = TruncatedSVD(n_components=100, random_state=0).fit_transform(np.log1p(CO))
Wn = normalize(W)
index = {w: i for i, w in enumerate(vocab)}


def nearest(word, k=6):
    if word not in index:
        return "(not in vocabulary)"
    sims = Wn @ Wn[index[word]]
    sims[index[word]] = -np.inf
    return ", ".join(f"{vocab[i]} {sims[i]:.3f}" for i in np.argsort(-sims)[:k])


for w in ("space", "doctor", "hockey", "car", "gun", "graphics"):
    print(f"{w:<10} -> {nearest(w)}")


Those are good, and nobody supplied a dictionary.

Now the famous party trick: subtract "man", add "woman", start from "king".


In [ ]:
def analogy(a, b, c, k=3):
    if not all(w in index for w in (a, b, c)):
        return "(a word is missing from the vocabulary)"
    v = Wn[index[b]] - Wn[index[a]] + Wn[index[c]]
    v = v / np.linalg.norm(v)
    sims = Wn @ v
    for w in (a, b, c):
        sims[index[w]] = -np.inf
    return ", ".join(f"{vocab[i]} {sims[i]:.3f}" for i in np.argsort(-sims)[:k])


for a, b, c in (("man", "woman", "king"), ("car", "engine", "computer")):
    print(f"{a} : {b} :: {c} : ?  ->  {analogy(a, b, c)}")


If the first line returned hockey players, the embedding is not broken. One of
the six newsgroups is `rec.sport.hockey`, and in that corpus a King is a team
from Los Angeles. The arithmetic worked correctly on the meaning it was given.

**An embedding is a map of the corpus it was trained on, not a map of meaning.**

That is the sentence to carry into Track 5. A retrieval system embeds your
documents and your users' questions into one space and returns whatever is
nearest. If the embedding model learned a word from one kind of text and your
documents mean it differently, retrieval quietly returns the wrong thing and the
generator writes a confident answer on top. Nothing raises an error.

## 5. What you have

- Similar items land near each other, without anyone labelling the arrangement.
- Compare by direction, not position, and normalise so the choice stops mattering.
- Very few dimensions do most of the work, and variance kept is not usefulness.
- The geometry is faithful to its corpus, including everything that corpus
  assumes.

Lesson 3.9 takes the other consequence: somebody has already paid to learn
representations like these, and you rarely need to start from nothing.
